In [99]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from gen_variable_standard_static import metrics_search_for_two_fragments_df
from tqdm import tqdm

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records'],
      dtype='str')

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [100]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

## Early practice

In [4]:
system_id = 1200
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)

In [5]:
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [6]:
test_df

,time,ac_power_kW_inverter,ac_power_kW_meter,year
0,2010-10-03 12:15:12,7.052258,NaN,2010
1,2010-10-03 12:30:12,9.621044,NaN,2010
2,2010-10-03 12:45:12,8.489045,NaN,2010
3,2010-10-03 13:00:13,9.352703,NaN,2010
4,2010-10-03 13:15:13,16.757570,NaN,2010
...,...,...,...,...
59796,2020-07-26 16:40:00,NaN,29.38,2020
59797,2020-07-26 16:45:00,NaN,28.30,2020
59798,2020-07-26 16:50:00,NaN,27.90,2020
59799,2020-07-26 16:55:00,NaN,27.06,2020


In [8]:
test_df.columns[test_df.columns.str.contains('inv')]

Index(['ac_power_kW_inverter'], dtype='str', name='')

In [87]:
def standardize(df: pd.DataFrame,
                null_or_zero: str,
                system_id: int,
                systems_cleaned: pd.DataFrame,
                drop_na: bool):
    '''Standardize input dataframes for univariate analysis.
    
    Parameters
    -----------
    df: pandas.DataFrame
        The dataframe output from ac_power_parquet_distiller_yearly.py
        or ac_power_parquet_distiller.py
    null_or_zero: str, "null", "zero", or "raw"
        Determine what to do with data less than 1 percent
        of maximum value
        If "null", replace small values by numpy.nan
        If "zero", replace with zero.
        If "raw", skip this cleaning step
        If anything else, throw a ValueError.
    system_id: int
        The system_id of the system.
    systems_cleaned: pd.DataFrame
        The metadata dataframe.
    drop_na: bool
        Determine whether or not to drop NA terms

    Returns
    ---------
    A list of one or two DataFrames.
    If both inverter and meter in input, get a list of two DataFrames
    If only one power input, get a list of one DataFrame
    Otherwise, bad input, ValueError
    '''
    # copy input for future use.
    df = df.copy(deep=True)
    # grab column names
    col_names = df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    # grab official max. value.
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_index = relevant_rows_systems.index[0]
    max_dc_capacity = relevant_rows_systems.loc[first_index, 'dc_capacity_kW']
    for pow_col_name in pow_col_names:
        local_max = df[pow_col_name].max()
        smaller_max = max(min(local_max, max_dc_capacity), 0)
        lower_bound = 0.01 * smaller_max
        if null_or_zero == 'null':
            df[pow_col_name] = df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else np.nan,  # not pandas.NA with float64 dtype
                axis = 1
            )
        elif null_or_zero == 'zero':
            df[pow_col_name] = df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else 0,
                axis = 1
            )
        elif null_or_zero == 'raw':
            pass
        else:
            raise ValueError(f'null_or_zero, input {null_or_zero}, is none of'
                             + '"null", "zero", or "raw".')
    # if both inverter and meter, make two columns
    if (
        any(pow_col_names.str.contains('inv'))
        and any(pow_col_names.str.contains('met'))
    ):
        # need two frames
        inv_pow_names = pow_col_names[pow_col_names.str.contains('inv')]
        if len(inv_pow_names) == 1:
            inv_pow_name = inv_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple inverter columns!')
        met_pow_names = pow_col_names[pow_col_names.str.contains('met')]
        if len(met_pow_names) == 1:
            met_pow_name = met_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple meter columns!')
        inv_data = df[['time', inv_pow_name]].copy(deep=True)
        met_data = df[['time', met_pow_name]].copy(deep=True)
        inv_data = inv_data.drop_duplicates()
        met_data = met_data.drop_duplicates()
        if drop_na:
            inv_data = inv_data.dropna()
            met_data = met_data.dropna()
        # rename columns
        renamer = {
            inv_pow_name: 'power',
            met_pow_name: 'power'
        }
        inv_data = inv_data.rename(columns=renamer)
        met_data = met_data.rename(columns=renamer)
        return [inv_data, met_data]
    elif len(pow_col_names) == 1:
        pow_col_name = pow_col_names[0]
        my_data = df[['time', pow_col_name]].copy(deep=True)
        renamer = {
            pow_col_name: 'power',
        }
        my_data = my_data.rename(columns=renamer)
        my_data = my_data.drop_duplicates()
        if drop_na:
            my_data = my_data.dropna()
        return [my_data,]
    else:
        raise ValueError('Not expected input!')

In [33]:
inv_out, met_out = standardize(test_df,
    'null',
    system_id,
    systems_cleaned,
    True
)

In [34]:
inv_out.describe()

,time,power
count,81103,81103.000000
mean,2012-08-13 19:14:24.518612480,16.438740
min,2010-10-03 12:15:12,0.486743
25%,2012-01-03 16:18:01,4.132064
50%,2012-07-15 06:35:31,12.693110
75%,2013-06-27 11:03:03,28.342121
max,2013-12-01 16:15:32,48.662719
std,NaN,13.346824


In [35]:
met_out.describe()

,time,power
count,146281,146281.000000
mean,2017-09-09 23:25:20.977242112,15.688420
min,2013-11-30 07:24:00,0.500000
25%,2014-06-04 08:33:00,3.560000
50%,2019-01-28 09:45:00,10.860000
75%,2019-10-17 12:05:00,27.340000
max,2020-07-26 17:00:00,48.160000
std,NaN,13.713706


In [36]:
inv_out['time'].diff().value_counts()

time
0 days 00:05:00    55545
0 days 00:15:00    13296
0 days 00:05:01     3944
0 days 00:15:01     2654
0 days 00:04:59     2001
                   ...  
0 days 14:50:02        1
0 days 15:04:58        1
0 days 15:40:00        1
0 days 15:14:55        1
0 days 17:09:56        1
Name: count, Length: 679, dtype: int64

In [80]:
inv_out['time'].diff().head()

0               NaT
1   0 days 00:15:00
2   0 days 00:15:00
3   0 days 00:15:01
4   0 days 00:15:00
Name: time, dtype: timedelta64[ns]

In [81]:
inv_out['time'].diff().tail()

36087   0 days 00:05:00
36090   0 days 00:05:00
36092   0 days 00:05:00
36095   0 days 00:05:00
36098   0 days 00:05:00
Name: time, dtype: timedelta64[ns]

In [39]:
met_out['time'].diff().value_counts()

time
0 days 00:05:00    91603
0 days 00:03:00    52386
0 days 00:04:36      328
0 days 00:05:24      325
0 days 00:10:00      118
                   ...  
0 days 00:05:22        1
0 days 00:04:38        1
0 days 00:05:45        1
0 days 00:04:15        1
0 days 02:10:00        1
Name: count, Length: 314, dtype: int64

In [77]:
met_out['time'].diff().head()

35213               NaT
35215   0 days 00:03:00
35216   0 days 00:03:00
35218   0 days 00:03:00
35220   0 days 00:03:00
Name: time, dtype: timedelta64[ns]

In [78]:
met_out['time'].diff().tail()

59796   0 days 00:05:00
59797   0 days 00:05:00
59798   0 days 00:05:00
59799   0 days 00:05:00
59800   0 days 00:05:00
Name: time, dtype: timedelta64[ns]

In [51]:
all_times_diff = test_df['time'].diff()
all_small_diff = all_times_diff[all_times_diff < timedelta(seconds=3600)]
all_small_diff.value_counts()

time
0 days 00:05:00    534056
0 days 00:03:00    376456
0 days 00:15:00     15515
0 days 00:05:01      4436
0 days 00:15:01      3083
                    ...  
0 days 00:46:44         1
0 days 00:03:16         1
0 days 00:05:51         1
0 days 00:04:09         1
0 days 00:05:21         1
Name: count, Length: 229, dtype: int64

In [50]:
test_df['time'].diff().value_counts()

time
0 days 00:05:00    534056
0 days 00:03:00    376456
0 days 00:15:00     15515
0 days 00:05:01      4436
0 days 00:15:01      3083
                    ...  
0 days 00:46:44         1
0 days 00:03:16         1
0 days 00:05:51         1
0 days 00:04:09         1
0 days 00:05:21         1
Name: count, Length: 752, dtype: int64

### Let me try a different system.  Could just be the off-and-on here

In [52]:
system_id = 4902
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)


In [53]:
nist_pq = pq.ParquetDataset(test_folder)
nist_df = nist_pq.read().to_pandas()

In [54]:
nist_df_times = nist_df['time'].diff()
nist_df_times.value_counts()

time
0 days 00:01:00    1871917
0 days 00:03:00       3438
0 days 00:02:00       2524
0 days 00:04:00        912
0 days 00:05:00        440
0 days 00:06:00        254
0 days 00:07:00        177
0 days 00:08:00        122
0 days 00:09:00         73
0 days 00:10:00         57
0 days 00:11:00         40
0 days 00:12:00         33
0 days 00:13:00         24
0 days 00:14:00         14
0 days 00:15:00         13
0 days 00:17:00         12
0 days 00:18:00          8
0 days 00:16:00          5
0 days 00:21:00          5
0 days 00:22:00          4
0 days 00:20:00          4
0 days 00:19:00          3
0 days 00:23:00          2
0 days 12:36:00          1
0 days 17:13:00          1
2 days 03:55:00          1
0 days 10:36:00          1
0 days 00:41:00          1
0 days 00:24:00          1
0 days 00:28:00          1
0 days 00:31:00          1
1 days 00:01:00          1
Name: count, dtype: int64

As expected, minute-by-minute updates.  Much nicer!

In [55]:
nist_df_late = nist_df[nist_df['time'] >= datetime(2017, 1, 1)]
nist_df_late_times = nist_df_late['time'].diff()
nist_df_late_times.value_counts() 

time
0 days 00:01:00    627696
0 days 00:02:00       322
0 days 00:03:00       164
0 days 00:04:00       140
0 days 00:05:00        81
0 days 00:07:00        27
0 days 00:06:00        22
0 days 00:08:00        20
0 days 00:09:00        11
0 days 00:10:00         8
0 days 00:13:00         6
0 days 00:11:00         5
0 days 00:12:00         4
0 days 00:17:00         2
0 days 00:14:00         1
0 days 00:18:00         1
0 days 00:15:00         1
Name: count, dtype: int64

### Another system

In [56]:
system_id = 4
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [57]:
test_df.head()

,time,ac_power_kW,year
0,2007-08-26 00:00:00,-0.000006,2007
1,2007-08-26 00:15:00,-0.000029,2007
2,2007-08-26 00:30:00,-0.000029,2007
3,2007-08-26 00:45:00,-0.000020,2007
4,2007-08-26 01:00:00,-0.000021,2007


In [58]:
good_data, = standardize(
    test_df,
    'null',
    system_id,
    systems_cleaned,
    True
)

In [59]:
good_data['time'].diff().value_counts()

time
0 days 00:01:00    2522600
0 days 00:15:00      24565
0 days 00:07:00        858
0 days 00:06:00        502
0 days 00:02:00        481
                    ...   
1 days 15:54:00          1
1 days 17:22:00          1
0 days 16:24:00          1
1 days 16:21:00          1
0 days 16:33:00          1
Name: count, Length: 750, dtype: int64

In [65]:
time_diffs = good_data['time'].diff()
short_diffs = time_diffs[time_diffs <= timedelta(seconds=3600)]
common_length = short_diffs.value_counts().index[0]

In [76]:
short_diffs.value_counts()

time
0 days 00:01:00    2522600
0 days 00:15:00      24565
0 days 00:07:00        858
0 days 00:06:00        502
0 days 00:02:00        481
0 days 00:08:00        322
0 days 00:03:00        300
0 days 00:04:00        192
0 days 00:05:00        143
0 days 00:30:00         80
0 days 00:09:00         73
0 days 00:10:00         67
0 days 00:14:00         49
0 days 00:11:00         49
0 days 00:13:00         48
0 days 00:12:00         44
0 days 00:21:00         42
0 days 00:16:00         37
0 days 00:20:00         32
0 days 00:17:00         30
0 days 00:45:00         27
0 days 00:18:00         27
0 days 00:22:00         25
0 days 00:27:00         25
0 days 00:19:00         23
0 days 00:24:00         22
0 days 00:23:00         22
0 days 00:26:00         21
0 days 00:25:00         21
0 days 00:28:00         21
0 days 00:29:00         19
0 days 00:32:00         17
0 days 00:31:00         16
0 days 00:33:00         14
0 days 00:47:00         13
0 days 00:42:00         12
0 days 00:53:00        

In [73]:
time_diffs

57                    NaT
58        0 days 00:15:00
59        0 days 00:15:00
60        0 days 00:15:00
61        0 days 00:15:00
                ...      
6342280   0 days 00:01:00
6342281   0 days 00:01:00
6342282   0 days 00:01:00
6342283   0 days 00:01:00
6342284   0 days 00:01:00
Name: time, Length: 2555751, dtype: timedelta64[ns]

In [84]:
time_diffs.name = 'Delta_t'

In [85]:
time_collect = pd.merge(left = test_df['time'], right = time_diffs, left_index=True, right_index=True)

In [86]:
time_collect

,time,Delta_t
57,2007-08-26 14:15:00,NaT
58,2007-08-26 14:30:00,0 days 00:15:00
59,2007-08-26 14:45:00,0 days 00:15:00
60,2007-08-26 15:00:00,0 days 00:15:00
61,2007-08-26 15:15:00,0 days 00:15:00
...,...,...
6342280,2023-02-28 17:13:00,0 days 00:01:00
6342281,2023-02-28 17:14:00,0 days 00:01:00
6342282,2023-02-28 17:15:00,0 days 00:01:00
6342283,2023-02-28 17:16:00,0 days 00:01:00


In [66]:
common_length

Timedelta('0 days 00:01:00')

In [72]:
short_diffs.head()

58   0 days 00:15:00
59   0 days 00:15:00
60   0 days 00:15:00
61   0 days 00:15:00
62   0 days 00:15:00
Name: time, dtype: timedelta64[ns]

In [67]:
common_length_ends = short_diffs[(short_diffs <= common_length + timedelta(seconds=1))
                                 & (short_diffs >= common_length - timedelta(seconds=1))]

In [71]:
common_length_ends.iloc[-5:-1]

6342280   0 days 00:01:00
6342281   0 days 00:01:00
6342282   0 days 00:01:00
6342283   0 days 00:01:00
Name: time, dtype: timedelta64[ns]

### Again!

In [88]:
system_id = 1300
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [89]:
test_df.head()

,time,ac_power_kW,year
0,2013-02-14 07:15:00,0.00000,2013
1,2013-02-14 07:20:00,0.00098,2013
2,2013-02-14 07:25:00,0.00915,2013
3,2013-02-14 07:30:00,0.02678,2013
4,2013-02-14 07:35:00,0.07955,2013


In [92]:
loc_data, = standardize(
    test_df,
    null_or_zero='null',
    system_id=1300,
    systems_cleaned=systems_cleaned,
    drop_na=True
)

In [93]:
loc_data.head()

,time,power
3,2013-02-14 07:30:00,0.02678
4,2013-02-14 07:35:00,0.07955
5,2013-02-14 07:40:00,0.12767
6,2013-02-14 07:45:00,0.12524
7,2013-02-14 07:50:00,0.06745


In [96]:
loc_data.loc[:, 'Delta_t'] = loc_data['time'].diff()

In [98]:
loc_data['Delta_t'].value_counts()

Delta_t
0 days 00:05:00     98344
0 days 00:10:00        81
0 days 00:15:00        39
0 days 11:15:00        39
0 days 11:40:00        29
                    ...  
0 days 02:05:00         1
28 days 12:05:00        1
0 days 02:40:00         1
0 days 02:20:00         1
0 days 22:30:00         1
Name: count, Length: 96, dtype: int64

In [104]:
def same_start_and_end_diff(
    system_id: int,
    systems_cleaned: pd.DataFrame
):
    # get and standardize data
    test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
    test_folder = Path(test_folder_str)
    test_pq = pq.ParquetDataset(test_folder)
    test_df = test_pq.read().to_pandas()
    my_data_list = standardize(
        test_df,
        null_or_zero='null',
        system_id=system_id,
        systems_cleaned=systems_cleaned,
        drop_na=True
    )
    if my_data_list is None or (len(my_data_list) == 0):
        raise ValueError('No data for this system!')
    out_list = []
    for df in my_data_list:
        early_data = df.iloc[0:1000]
        early_data_time_diffs = early_data['time'].diff()
        common_early_time = early_data_time_diffs.value_counts().index[0]
        late_data = df.iloc[-1000:]
        late_data_time_diffs = late_data['time'].diff()
        common_late_time = late_data_time_diffs.value_counts().index[0]
        out_list.append(
            ((common_early_time - common_late_time) < timedelta(seconds=5))
            and ((common_late_time - common_early_time) < timedelta(seconds=5))
        )
    return out_list

In [106]:
data_dict = dict()
for system_id in tqdm(enough_data_parquet_power_list):
    test_results = same_start_and_end_diff(
        system_id=system_id, systems_cleaned=systems_cleaned
    )
    if len(test_results) == 2:
        data_dict[(system_id, 'inverter')] = test_results[0]
        data_dict[(system_id, 'meter')] = test_results[1]
    elif len(test_results) == 1:
        data_dict[(system_id, 'unknown')] = test_results[0]
    else:
        raise RuntimeError('Bad results from same_start_and_end_diff(*args)!')
out_df = pd.Series(data = data_dict, name='start_and_end_agreement')

100%|██████████| 82/82 [29:41<00:00, 21.73s/it]  


In [107]:
out_df.head()

4   unknown    False
10  unknown    False
33  unknown     True
34  unknown     True
35  unknown     True
Name: start_and_end_agreement, dtype: bool

In [109]:
out_df.index

MultiIndex([(   4,  'unknown'),
            (  10,  'unknown'),
            (  33,  'unknown'),
            (  34,  'unknown'),
            (  35,  'unknown'),
            (  36,  'unknown'),
            (  50,  'unknown'),
            (  51,  'unknown'),
            (1199,  'unknown'),
            (1200, 'inverter'),
            (1200,    'meter'),
            (1201,  'unknown'),
            (1202, 'inverter'),
            (1202,    'meter'),
            (1203, 'inverter'),
            (1203,    'meter'),
            (1204,  'unknown'),
            (1208, 'inverter'),
            (1208,    'meter'),
            (1214,  'unknown'),
            (1217,  'unknown'),
            (1219,  'unknown'),
            (1221,  'unknown'),
            (1223,  'unknown'),
            (1225,  'unknown'),
            (1231,  'unknown'),
            (1239,  'unknown'),
            (1244,  'unknown'),
            (1246,  'unknown'),
            (1248,  'unknown'),
            (1249,  'unknown'),
        

In [110]:
out_df.index.names = ['system_id', 'source_type']

In [111]:
out_df

system_id  source_type
4          unknown        False
10         unknown        False
33         unknown         True
34         unknown         True
35         unknown         True
                          ...  
4901       meter           True
4902       inverter        True
           meter           True
4903       inverter        True
           meter           True
Name: start_and_end_agreement, Length: 91, dtype: bool

In [112]:
out_df.columns = ['consistent_start_and_end']

In [115]:
out_df.head()

system_id  source_type
4          unknown        False
10         unknown        False
33         unknown         True
34         unknown         True
35         unknown         True
Name: start_and_end_agreement, dtype: bool

In [116]:
out_df.value_counts()

start_and_end_agreement
True     76
False    15
Name: count, dtype: int64

In [117]:
out_df[~out_df]

system_id  source_type
4          unknown        False
10         unknown        False
50         unknown        False
51         unknown        False
1199       unknown        False
1200       inverter       False
           meter          False
1201       unknown        False
1202       meter          False
1203       inverter       False
           meter          False
1308       unknown        False
1418       unknown        False
1419       unknown        False
1420       unknown        False
Name: start_and_end_agreement, dtype: bool